# Práctica 1 — Optimización Estocástica

**Ruteo urbano con tiempos de viaje estocásticos en Manhattan**

Universidad EAFIT · Maestría en Matemáticas Aplicadas · 2026-2

Programa estocástico lineal de dos etapas con primera etapa binaria (ruteo) y
segunda etapa continua (recurso). Se construye una aproximación por promedio
muestral (SAA), se valida la forma extensa MILP contra una implementación propia
del algoritmo *Integer L-shaped*, y se contrasta la solución estocástica con la
aproximación determinista del perfil promedio de tiempos.

**Nota sobre el solver.** Esta versión del notebook usa `scipy.optimize`
(HiGHS) en vez de Gurobi, para no depender de una licencia con límite de
tamaño ni de estar en la red de la universidad. El subproblema de segunda
etapa se evalúa con la forma cerrada del recurso (ver más abajo), así que ni
siquiera hace falta resolver un LP por escenario.

## 0. Preparación del entorno

In [ ]:
!pip install -q scipy polars pyarrow matplotlib

### Módulos del proyecto

Las tres celdas siguientes escriben los módulos en el entorno de ejecución, de
modo que el notebook sea autocontenido y reproducible con *Run all*.

- `p1_datos.py`: descarga, filtros y construcción de los pools origen–destino.
- `p1_modelo_scipy.py`: forma extensa MILP, subproblemas de recurso (forma
  cerrada) e Integer L-shaped, todo con scipy/HiGHS.
- `p1_experimentos.py`: orquestación de los experimentos y visualizaciones.

In [ ]:
%%writefile p1_datos.py
"""Construcción de pools origen-destino a partir de los registros de la TLC.

El mismo procedimiento se aplica a enero (entrenamiento) y a febrero
(validación temporal fuera de muestra), sin mezclar los pools.
"""

import urllib.request
from datetime import datetime, time

import polars as pl

KAPPA = 1.00        # USD/km
MILLA_A_KM = 1.60934

ZONAS = [
    (0, "Depósito: Javits Center", 40.75750, -74.00250, 246),
    (1, "Times Square", 40.75800, -73.98550, 230),
    (2, "Rockefeller Center", 40.75870, -73.97870, 161),
    (3, "Grand Central Terminal", 40.75278, -73.97722, 162),
    (4, "New York Public Library (Main)", 40.75306, -73.98194, 164),
    (5, "Union Square", 40.73590, -73.99110, 234),
    (6, "Washington Square Park", 40.73083, -73.99750, 114),
    (7, "Madison Square Garden", 40.75056, -73.99361, 186),
    (8, "One World Trade Center", 40.71274, -74.01338, 261),
    (9, "New York Stock Exchange", 40.70693, -74.01125, 87),
    (10, "South Street Seaport (Pier 17)", 40.70600, -74.00270, 209),
]

ESQUEMA_ZONAS = {
    "i": pl.Int64, "Sitio": pl.Utf8, "Latitud": pl.Float64,
    "Longitud": pl.Float64, "LocationID": pl.Int64,
}

URL_ENERO = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2015-01.parquet"
URL_FEBRERO = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2015-02.parquet"


def zonas_dataframe():
    return pl.DataFrame(ZONAS, schema=ESQUEMA_ZONAS, orient="row")


def descargar(url, destino=None):
    destino = destino or url.rsplit("/", 1)[-1]
    urllib.request.urlretrieve(url, destino)
    return destino


def construir_pools(archivo, inicio, fin_exclusivo, zonas_df=None):
    """Aplica los cuatro filtros del enunciado y devuelve los pools por arco.

    El límite superior es EXCLUSIVO: para enero se pasa 2015-02-01 y para
    febrero 2015-03-01, de modo que el último día del mes queda incluido.
    """
    zonas_df = zonas_dataframe() if zonas_df is None else zonas_df
    loc_ids = zonas_df["LocationID"].to_list()

    datos = (
        pl.scan_parquet(archivo)
        .select(["tpep_pickup_datetime", "tpep_dropoff_datetime",
                 "PULocationID", "DOLocationID", "trip_distance"])
        # 1) ventana de fechas, lunes a viernes, franja [09:00, 17:00)
        .filter(
            pl.col("tpep_pickup_datetime") >= inicio,
            pl.col("tpep_pickup_datetime") < fin_exclusivo,
            pl.col("tpep_pickup_datetime").dt.weekday().is_between(1, 5),
            pl.col("tpep_pickup_datetime").dt.time() >= time(9, 0),
            pl.col("tpep_pickup_datetime").dt.time() < time(17, 0),
        )
        # 2) origen y destino en zonas distintas de la tabla
        .filter(
            pl.col("PULocationID").is_in(loc_ids),
            pl.col("DOLocationID").is_in(loc_ids),
            pl.col("PULocationID") != pl.col("DOLocationID"),
        )
        # 3) duración en minutos dentro de [1, 90]
        .with_columns(
            ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
             .dt.total_seconds() / 60.0).alias("t")
        )
        .filter(pl.col("t").is_between(1, 90))
        # 4) distancia entre 0.1 y 30 millas
        .filter(pl.col("trip_distance").is_between(0.1, 30))
        .collect()
    )

    pools = (
        datos.group_by(["PULocationID", "DOLocationID"])
        .agg(
            pl.col("t").alias("P_ij"),
            pl.len().alias("registros"),
            (pl.col("trip_distance").median() * MILLA_A_KM).alias("d_road_ij"),
        )
        .with_columns((KAPPA * pl.col("d_road_ij")).alias("c_ij"))
        .join(zonas_df.select([pl.col("LocationID").alias("PULocationID"), pl.col("i")]),
              on="PULocationID")
        .join(zonas_df.select([pl.col("LocationID").alias("DOLocationID"),
                               pl.col("i").alias("j")]),
              on="DOLocationID")
        .select(["i", "j", "d_road_ij", "c_ij", "P_ij", "registros"])
        .sort(["i", "j"])
    )
    return datos, pools


def control_calidad(pools, n_arcos_esperado=110, min_registros_esperado=None):
    n = pools.shape[0]
    minimo = int(pools["registros"].min())
    print(f"Número de pools: |A| = {n} (esperado {n_arcos_esperado})")
    print(f"Pool más pequeño: {minimo} registros", end="")
    if min_registros_esperado is not None:
        print(f" (esperado {min_registros_esperado})")
    else:
        print()
    ok = n == n_arcos_esperado
    if min_registros_esperado is not None:
        ok = ok and minimo == min_registros_esperado
    print("Control de calidad:", "OK" if ok else "REVISAR")
    return ok


def pools_enero(archivo=None):
    archivo = archivo or descargar(URL_ENERO)
    return construir_pools(archivo, datetime(2015, 1, 1), datetime(2015, 2, 1))


def pools_febrero(archivo=None):
    archivo = archivo or descargar(URL_FEBRERO)
    return construir_pools(archivo, datetime(2015, 2, 1), datetime(2015, 3, 1))


def alinear_pools(pools_ref, pools_nuevos):
    """Reordena pools_nuevos según el orden de arcos de pools_ref.

    Las distancias y los costos c_ij se mantienen fijos en los de entrenamiento,
    tal como exige el enunciado para la validación temporal.
    """
    orden = pools_ref.select(["i", "j", "d_road_ij", "c_ij"])
    return (orden.join(pools_nuevos.select(["i", "j", "P_ij", "registros"]),
                       on=["i", "j"], how="left")
            .select(["i", "j", "d_road_ij", "c_ij", "P_ij", "registros"])
            .sort(["i", "j"]))

In [ ]:
%%writefile p1_modelo_scipy.py
"""Práctica 1 - Optimización Estocástica (EAFIT, 2026-2).

Reimplementación de ``p1_modelo.py`` SIN GUROBI: usa exclusivamente
``scipy.optimize`` (HiGHS, que scipy trae integrado desde la versión 1.9) y
``numpy``. Es un reemplazo directo -- mismo API público, mismos nombres de
clases, funciones y campos en los diccionarios de resultado -- pensado para
poder sustituir la línea

    from p1_modelo import (...)

por

    from p1_modelo_scipy import (...)

sin tocar ``p1_experimentos.py`` ni el notebook.

Motivación
----------
La licencia restringida que viene con ``pip install gurobipy`` limita el
tamaño de los modelos (variables + restricciones). El README del proyecto
asumía que esto solo afectaba a la forma extensa con K=200, pero en la
práctica el LICENSE LIMIT también se alcanza en el maestro del Integer
L-shaped, porque los cortes de Benders y los cortes enteros se van
acumulando como restricciones nuevas sobre el MISMO modelo persistente
durante todo el branch-and-bound (el maestro que revienta con
``GurobiError: Model too large for size-limited license`` incluso con
K=50). Conseguir una licencia académica sin restricción de tamaño (WLS)
requiere estar en la red de la universidad, algo que no siempre es posible.

Este módulo elimina la dependencia de Gurobi por completo:

* La forma extensa y el maestro del Integer L-shaped se resuelven con
  ``scipy.optimize.milp`` / ``scipy.optimize.linprog``, que usan HiGHS
  (Apache 2.0, sin límite de tamaño ni de licencia).
* El subproblema de segunda etapa YA NO SE RESUELVE COMO LP. El propio
  informe documenta que, como alpha_i = 1 para todo sitio, el recurso
  Q(x, xi_s) = phi(tau_s) tiene forma cerrada (``recurso_analitico``),
  validada contra el LP hasta precisión de máquina. Aquí esa forma cerrada
  deja de ser solo una verificación y pasa a ser el motor de evaluación:
  no hace falta resolver K LPs (ni con Gurobi ni con HiGHS) para obtener
  Q_K(x), los cortes de Benders (con su dual exacto) ni los detalles de
  asignación (o, e, r, u) que necesitan los diagnósticos y las figuras.
  Esto es exacto (no una aproximación) y, de paso, es la parte más costosa
  del algoritmo original, así que el Integer L-shaped queda además más
  rápido que la versión con Gurobi.

La única pieza que de verdad necesita un solver LP/MILP es entonces el
maestro (LP pequeño, resuelto muchas veces) y la forma extensa (un MILP
grande solo con K=50/200, usada una vez por experimento). Ambas usan HiGHS
vía scipy, sin restricciones de tamaño.
"""

import heapq
import time

import numpy as np
from scipy import sparse
from scipy.optimize import Bounds, LinearConstraint, linprog, milp

# --------------------------------------------------------------------------
# Parámetros del experimento (sección 3 del enunciado) -- idénticos a
# p1_modelo.py, no dependen del solver.
# --------------------------------------------------------------------------
H = 390.0          # horizonte operativo regular [min]
O_BAR = 30.0       # tiempo adicional ordinario máximo [min]
C_OT = 1.50        # costo del tiempo adicional ordinario [USD/min]
C_EM = 6.00        # costo del sobretiempo de emergencia [USD/min]
KAPPA = 1.00       # costo de desplazamiento [USD/km]

N_SITIOS = 10
SITIOS = list(range(1, N_SITIOS + 1))
NODOS = list(range(0, N_SITIOS + 1))

DEMANDA = {1: 30.0, 2: 30.0, 3: 35.0, 4: 30.0, 5: 25.0,
           6: 25.0, 7: 35.0, 8: 40.0, 9: 35.0, 10: 25.0}

C_OUT = {1: 2.60, 2: 2.40, 3: 2.80, 4: 2.30, 5: 2.10,
         6: 2.00, 7: 2.70, 8: 3.00, 9: 2.90, 10: 2.20}

ALPHA = {i: 1.0 for i in SITIOS}

NOMBRES = {
    0: "Depósito: Javits Center", 1: "Times Square", 2: "Rockefeller Center",
    3: "Grand Central Terminal", 4: "New York Public Library", 5: "Union Square",
    6: "Washington Square Park", 7: "Madison Square Garden",
    8: "One World Trade Center", 9: "New York Stock Exchange",
    10: "South Street Seaport",
}


# --------------------------------------------------------------------------
# Instancia SAA -- sin cambios respecto a p1_modelo.py, no usa el solver.
# --------------------------------------------------------------------------
class Instancia:
    """Arcos, costos de primera etapa y matriz de escenarios (K x |A|)."""

    def __init__(self, arcos, c, xi):
        self.arcos = [tuple(a) for a in arcos]
        self.nA = len(self.arcos)
        self.c = np.asarray(c, dtype=float)
        self.xi = np.atleast_2d(np.asarray(xi, dtype=float))
        self.K = self.xi.shape[0]
        self.pos = {a: k for k, a in enumerate(self.arcos)}
        if self.xi.shape[1] != self.nA:
            raise ValueError("xi debe tener |A| columnas")

    def promedio(self):
        """Instancia de un solo escenario con el perfil promedio de tiempos."""
        return Instancia(self.arcos, self.c, self.xi.mean(axis=0, keepdims=True))


def pools_a_numpy(pools_df):
    """Convierte el dataframe de pools (polars) en arcos, costos y pools.

    Espera las columnas i, j, c_ij y P_ij tal como las produce el notebook.
    """
    arcos, costos, pools = [], [], {}
    for fila in pools_df.iter_rows(named=True):
        arco = (int(fila["i"]), int(fila["j"]))
        arcos.append(arco)
        costos.append(float(fila["c_ij"]))
        pools[arco] = np.asarray(fila["P_ij"], dtype=float)
    return arcos, np.asarray(costos, dtype=float), pools


def muestrear_escenarios(arcos, pools, K, seed):
    """Muestrea K escenarios con reemplazo, de forma independiente entre arcos.

    Se usa un único generador de NumPy consumido arco por arco: esto garantiza
    reproducibilidad sin reutilizar la misma semilla en cada arco, que es lo que
    induciría dependencia entre componentes de xi.
    """
    rng = np.random.default_rng(seed)
    xi = np.empty((K, len(arcos)), dtype=float)
    for k, arco in enumerate(arcos):
        pool = pools[arco]
        xi[:, k] = pool[rng.integers(0, len(pool), size=K)]
    return xi


def construir_instancia(pools_df, K, seed):
    arcos, costos, pools = pools_a_numpy(pools_df)
    return Instancia(arcos, costos, muestrear_escenarios(arcos, pools, K, seed))


def extraer_ruta(inst, x_vec):
    """Devuelve la secuencia de nodos 0 -> ... -> 0 de una solución binaria."""
    siguiente = {}
    for k, (i, j) in enumerate(inst.arcos):
        if x_vec[k] > 0.5:
            siguiente[i] = j
    ruta, actual = [0], 0
    for _ in range(len(NODOS)):
        actual = siguiente.get(actual)
        if actual is None:
            return None
        ruta.append(actual)
        if actual == 0:
            return ruta if len(ruta) == len(NODOS) + 1 else None
    return None


# --------------------------------------------------------------------------
# Forma extensa MILP (scipy.optimize.milp / HiGHS)
# --------------------------------------------------------------------------
def resolver_extenso(inst, tiempo_limite=1800, mip_gap=1e-6, verbose=False):
    """Resuelve el equivalente determinista completo de la muestra SAA."""
    t0 = time.time()
    nA, nS, K = inst.nA, N_SITIOS, inst.K
    bloque = 2 * nS + 2  # columnas u, r, o, e de cada escenario
    off_v = nA
    off_esc = nA + nS
    n_total = off_esc + K * bloque

    def off_u(s):
        return off_esc + s * bloque

    def off_r(s):
        return off_esc + s * bloque + nS

    def off_o(s):
        return off_esc + s * bloque + 2 * nS

    def off_e(s):
        return off_esc + s * bloque + 2 * nS + 1

    idx_x = {a: k for k, a in enumerate(inst.arcos)}
    idx_v = {i: off_v + t for t, i in enumerate(SITIOS)}

    filas, cols, vals, lo, hi = [], [], [], [], []
    r = 0
    # grado de entrada/salida
    for i in NODOS:
        for j in NODOS:
            if j != i:
                filas.append(r); cols.append(idx_x[(i, j)]); vals.append(1.0)
        lo.append(1.0); hi.append(1.0); r += 1
        for j in NODOS:
            if j != i:
                filas.append(r); cols.append(idx_x[(j, i)]); vals.append(1.0)
        lo.append(1.0); hi.append(1.0); r += 1
    # Miller-Tucker-Zemlin
    for i in SITIOS:
        for j in SITIOS:
            if i != j:
                filas += [r, r, r]
                cols += [idx_v[i], idx_v[j], idx_x[(i, j)]]
                vals += [1.0, -1.0, float(nS)]
                lo.append(-np.inf); hi.append(float(nS - 1)); r += 1
    # demanda y balance de tiempo por escenario
    for s in range(K):
        for t, i in enumerate(SITIOS):
            filas += [r, r]
            cols += [off_u(s) + t, off_r(s) + t]
            vals += [1.0, 1.0]
            lo.append(DEMANDA[i]); hi.append(DEMANDA[i]); r += 1
        for k, a in enumerate(inst.arcos):
            xi_sk = float(inst.xi[s, k])
            if xi_sk != 0.0:
                filas.append(r); cols.append(idx_x[a]); vals.append(xi_sk)
        for t, i in enumerate(SITIOS):
            filas.append(r); cols.append(off_u(s) + t); vals.append(float(ALPHA[i]))
        filas.append(r); cols.append(off_o(s)); vals.append(-1.0)
        filas.append(r); cols.append(off_e(s)); vals.append(-1.0)
        lo.append(-np.inf); hi.append(H); r += 1

    A = sparse.csr_matrix((vals, (filas, cols)), shape=(r, n_total))
    constr = LinearConstraint(A, np.asarray(lo), np.asarray(hi))

    c_obj = np.zeros(n_total)
    for k, a in enumerate(inst.arcos):
        c_obj[idx_x[a]] = inst.c[k]
    for s in range(K):
        for t, i in enumerate(SITIOS):
            c_obj[off_r(s) + t] = C_OUT[i] / K
        c_obj[off_o(s)] = C_OT / K
        c_obj[off_e(s)] = C_EM / K

    lb = np.zeros(n_total)
    ub = np.full(n_total, np.inf)
    for a in inst.arcos:
        ub[idx_x[a]] = 1.0
    for i in SITIOS:
        lb[idx_v[i]] = 1.0
        ub[idx_v[i]] = float(nS)
    for s in range(K):
        ub[off_o(s)] = O_BAR

    integrality = np.zeros(n_total)
    for a in inst.arcos:
        integrality[idx_x[a]] = 1

    res = milp(c_obj, integrality=integrality, bounds=Bounds(lb, ub),
               constraints=[constr],
               options={"mip_rel_gap": mip_gap, "time_limit": tiempo_limite,
                        "disp": verbose})
    if not res.success or res.x is None:
        raise RuntimeError(
            f"La forma extensa no encontró solución factible "
            f"(status {res.status}: {res.message}).")

    x_vec = np.array([res.x[idx_x[a]] for a in inst.arcos])
    return {
        "x": np.round(x_vec),
        "obj": float(res.fun),
        "cota": float(res.mip_dual_bound),
        "gap": float(res.mip_gap),
        "tiempo": time.time() - t0,
        "n_vars": n_total,
        "n_cons": r,
        "n_bin": nA,
        "ruta": extraer_ruta(inst, x_vec),
    }


# --------------------------------------------------------------------------
# Segunda etapa: forma cerrada (sin LP)
# --------------------------------------------------------------------------
D_TOTAL = sum(DEMANDA.values())

# Escalera de costos marginales: el tiempo adicional ordinario es lo más
# barato, luego la tercerización por sitio en orden creciente de c_out, y al
# final la emergencia, que es más cara que cualquier tercerización y no
# tiene cota. Se usa para Q(tau) (recurso_analitico) y, en paralelo, para
# reconstruir la asignación óptima (o, e, r, u) en _asignar_recurso.
_TRAMOS = ([(C_OT, O_BAR)]
           + sorted((C_OUT[i], DEMANDA[i]) for i in SITIOS)
           + [(C_EM, np.inf)])
_CAP_ACUM = np.cumsum([cap for _, cap in _TRAMOS])
_COSTO_TRAMO = np.array([costo for costo, _ in _TRAMOS])

_ORDEN_TRAMOS = ([("O", O_BAR)]
                  + [(i, DEMANDA[i]) for i in sorted(SITIOS, key=lambda i: C_OUT[i])]
                  + [("E", np.inf)])


def recurso_analitico(tau):
    """Evalúa Q(x, xi_s) en forma cerrada a partir del escalar tau.

    El subproblema de segunda etapa no depende del escenario salvo por
    tau_s = sum_ij xi_ij^(s) x_ij, de modo que Q(x, xi_s) = phi(tau_s) para una
    única función convexa lineal por tramos phi. Coincide con el óptimo del
    LP y con su dual hasta precisión de máquina (verificado en
    ``test_modelo_scipy.py`` contra ``scipy.optimize.linprog``).

    Solo es válida con alpha_i = 1 para todo sitio.
    """
    if any(abs(ALPHA[i] - 1.0) > 1e-12 for i in SITIOS):
        raise ValueError("recurso_analitico supone alpha_i = 1")

    tau = np.atleast_1d(np.asarray(tau, dtype=float))
    deficit = np.maximum(tau + D_TOTAL - H, 0.0)

    Q = np.zeros_like(deficit)
    restante = deficit.copy()
    for costo, cap in _TRAMOS:
        usado = np.minimum(restante, cap)
        Q += costo * usado
        restante -= usado

    tramo = np.searchsorted(_CAP_ACUM, deficit, side="left")
    tramo = np.clip(tramo, 0, len(_TRAMOS) - 1)
    marginal = np.where(deficit > 0, _COSTO_TRAMO[tramo], 0.0)
    return Q, marginal


def _asignar_recurso(tau):
    """Reconstruye la asignación óptima (o, e, r, u) para cada escenario.

    Sigue la misma escalera de tramos que ``recurso_analitico``, de modo que
    la suma ponderada por costos coincide exactamente con Q(tau). Devuelve
    o, e como arreglos (K,) y r, u como arreglos (K, N_SITIOS) en el orden
    de ``SITIOS`` (no en el orden de la escalera, para que sean directamente
    indexables por sitio como en p1_modelo.py).
    """
    tau = np.atleast_1d(np.asarray(tau, dtype=float))
    K = tau.shape[0]
    deficit = np.maximum(tau + D_TOTAL - H, 0.0)
    restante = deficit.copy()

    o = np.zeros(K)
    e = np.zeros(K)
    r_por_sitio = {i: np.zeros(K) for i in SITIOS}
    for etiqueta, cap in _ORDEN_TRAMOS:
        usado = np.minimum(restante, cap)
        if etiqueta == "O":
            o = usado
        elif etiqueta == "E":
            e = usado
        else:
            r_por_sitio[etiqueta] = usado
        restante = restante - usado

    r = np.column_stack([r_por_sitio[i] for i in SITIOS])
    u = np.column_stack([np.full(K, DEMANDA[i]) for i in SITIOS]) - r
    return o, e, r, u


class EvaluadorRecurso:
    """Evalúa el recurso de segunda etapa con la forma cerrada del recurso.

    En p1_modelo.py esta clase mantenía K modelos LP persistentes (uno por
    escenario) y solo actualizaba su lado derecho en cada llamada. Aquí no
    hace falta: Q(x, xi_s) = phi(tau_s) es exacta (ver ``recurso_analitico``
    y el módulo docstring), así que evaluar K escenarios es una operación
    vectorizada de numpy, sin resolver ningún LP. El resultado -- valores,
    dual y corte agregado -- es idéntico al que producía el LP de Gurobi,
    hasta precisión de máquina (verificado en ``test_modelo_scipy.py``).
    """

    def __init__(self, inst):
        self.inst = inst
        self.n_lps = 0  # se conserva por compatibilidad con el API original

    def evaluar(self, x_vec, detalle=False):
        """Devuelve Q_K(x), el corte agregado y los valores por escenario.

        El corte es la pareja (const, coef) tal que  theta >= const + coef @ x,
        igual que en p1_modelo.py: viene de la desigualdad de subgradiente
            Q_s(tau) >= Q_s(tau^v) - lambda_s (tau - tau^v),
        promediada sobre los K escenarios con probabilidad 1/K, usando
        lambda_s = -marginal_s (el dual exacto de la restricción de tiempo,
        igual signo que ``con_tiempo[s].Pi`` en el modelo de Gurobi).
        """
        x_vec = np.asarray(x_vec, dtype=float)
        tau = self.inst.xi @ x_vec
        Q, marginal = recurso_analitico(tau)
        lam = -marginal
        self.n_lps += self.inst.K

        const = float(np.mean(Q + lam * tau))
        coef = -(self.inst.xi * lam[:, None]).mean(axis=0)

        info = None
        if detalle:
            o_s, e_s, r_s, u_s = _asignar_recurso(tau)
            info = [{
                "Q": float(Q[s]),
                "tau": float(tau[s]),
                "o": float(o_s[s]),
                "e": float(e_s[s]),
                "r": r_s[s],
                "u": u_s[s],
            } for s in range(self.inst.K)]
        return float(Q.mean()), (const, coef), Q, info


# --------------------------------------------------------------------------
# Maestro del Integer L-shaped (scipy.optimize.linprog / HiGHS)
# --------------------------------------------------------------------------
class MaestroRuteo:
    """Maestro LP por nodo: ruteo relajado, theta >= L y cortes globales.

    A diferencia de p1_modelo.py (un ``gp.Model`` persistente al que se le
    añadían restricciones), aquí se guardan las filas de los cortes en
    listas y se ensambla la matriz dispersa completa en cada llamada a
    ``resolver``. Es más simple de razonar y no hay límite de tamaño de
    licencia; con los tamaños de esta práctica (maestro con ~120 columnas)
    el costo de reensamblar es insignificante frente al propio LP.
    """

    def __init__(self, inst, theta_lb=0.0):
        self.inst = inst
        self.nA = inst.nA
        self.off_v = self.nA
        self.off_theta = self.nA + N_SITIOS
        self.n_total = self.off_theta + 1
        self.theta_lb = theta_lb

        self.idx_x = {a: k for k, a in enumerate(inst.arcos)}
        self.idx_v = {i: self.off_v + t for t, i in enumerate(SITIOS)}

        # Bloque estático de ruteo (grado + MTZ), construido una sola vez.
        filas, cols, vals, lo, hi = [], [], [], [], []
        r = 0
        for i in NODOS:
            for j in NODOS:
                if j != i:
                    filas.append(r); cols.append(self.idx_x[(i, j)]); vals.append(1.0)
            lo.append(1.0); hi.append(1.0); r += 1
            for j in NODOS:
                if j != i:
                    filas.append(r); cols.append(self.idx_x[(j, i)]); vals.append(1.0)
            lo.append(1.0); hi.append(1.0); r += 1
        for i in SITIOS:
            for j in SITIOS:
                if i != j:
                    filas += [r, r, r]
                    cols += [self.idx_v[i], self.idx_v[j], self.idx_x[(i, j)]]
                    vals += [1.0, -1.0, float(N_SITIOS)]
                    lo.append(-np.inf); hi.append(float(N_SITIOS - 1)); r += 1
        self._filas_est, self._cols_est, self._vals_est = filas, cols, vals
        self._lo_est, self._hi_est = lo, hi
        self._n_est = r

        c_obj = np.zeros(self.n_total)
        for k, a in enumerate(inst.arcos):
            c_obj[k] = inst.c[k]
        c_obj[self.off_theta] = 1.0
        self.c_obj = c_obj

        # Cortes dinámicos (Benders + enteros), como filas nuevas.
        self._filas_din, self._cols_din, self._vals_din, self._rhs_din = [], [], [], []
        self.n_benders = 0
        self.n_enteros = 0

    def agregar_corte_benders(self, corte):
        """theta >= const + coef @ x  <=>  -theta + coef @ x <= -const."""
        const, coef = corte
        # Offset por self._n_est: las filas del bloque estático (grado + MTZ)
        # ya ocupan las filas [0, self._n_est); sin este offset, la fila del
        # corte colisionaría (mismo índice de fila) con una fila estática y
        # scipy.sparse sumaría ambas al construir la matriz, corrompiendo esa
        # restricción en vez de añadir el corte.
        r = self._n_est + self.n_benders + self.n_enteros
        self._filas_din.append(r); self._cols_din.append(self.off_theta)
        self._vals_din.append(-1.0)
        for k in range(self.nA):
            ck = float(coef[k])
            if ck != 0.0:
                self._filas_din.append(r); self._cols_din.append(k)
                self._vals_din.append(ck)
        self._rhs_din.append(-float(const))
        self.n_benders += 1

    def agregar_corte_entero(self, x_bin, Qx, L=0.0):
        """Corte de optimalidad entero de Laporte y Louveaux.

        theta >= (Q - L) (sum_{S} x - sum_{no S} x - |S| + 1) + L,
        con S el conjunto de arcos activos en x_bin. Es ajustado en x_bin y
        se reduce a theta >= L en cualquier otro punto binario. En forma
        <=:  -theta + peso*sum_S x - peso*sum_{no S} x <= peso*(|S|-1) - L.
        """
        S = [k for k in range(self.nA) if x_bin[k] > 0.5]
        fuera = [k for k in range(self.nA) if x_bin[k] <= 0.5]
        peso = Qx - L
        r = self._n_est + self.n_benders + self.n_enteros
        self._filas_din.append(r); self._cols_din.append(self.off_theta)
        self._vals_din.append(-1.0)
        for k in S:
            self._filas_din.append(r); self._cols_din.append(k)
            self._vals_din.append(peso)
        for k in fuera:
            self._filas_din.append(r); self._cols_din.append(k)
            self._vals_din.append(-peso)
        self._rhs_din.append(peso * (len(S) - 1) - L)
        self.n_enteros += 1

    def resolver(self, lb_v, ub_v):
        n_cortes = self.n_benders + self.n_enteros
        n_filas = self._n_est + n_cortes
        filas = self._filas_est + self._filas_din
        cols = self._cols_est + self._cols_din
        vals = self._vals_est + self._vals_din
        lo = np.asarray(self._lo_est + [-np.inf] * n_cortes)
        hi = np.asarray(self._hi_est + self._rhs_din)

        A = sparse.csr_matrix((vals, (filas, cols)), shape=(n_filas, self.n_total))

        eq_mask = lo == hi
        A_eq = A[eq_mask] if np.any(eq_mask) else None
        b_eq = hi[eq_mask] if np.any(eq_mask) else None
        A_ub = A[~eq_mask] if np.any(~eq_mask) else None
        b_ub = hi[~eq_mask] if np.any(~eq_mask) else None

        lb = np.zeros(self.n_total)
        ub = np.full(self.n_total, np.inf)
        for k in range(self.nA):
            lb[k] = lb_v[k]
            ub[k] = ub_v[k]
        for i in SITIOS:
            lb[self.idx_v[i]] = 1.0
            ub[self.idx_v[i]] = float(N_SITIOS)
        lb[self.off_theta] = self.theta_lb

        res = linprog(self.c_obj, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                       bounds=list(zip(lb, ub)), method="highs")
        if res.status != 0:
            return None
        x_vec = res.x[:self.nA]
        theta = float(res.x[self.off_theta])
        return x_vec, theta, float(res.fun)


# --------------------------------------------------------------------------
# Integer L-shaped -- lógica de branch-and-bound idéntica a p1_modelo.py,
# no depende del solver salvo a través de MaestroRuteo/EvaluadorRecurso.
# --------------------------------------------------------------------------
class IntegerLShaped:
    """Branch-and-bound sobre los arcos con descomposición L-shaped por nodo.

    Selección de nodo por mejor cota, ramificación sobre la variable de arco más
    cercana a 0.5 y criterio de parada  (UB - LB) / max(1, |UB|) <= tol_gap.
    """

    def __init__(self, inst, L=0.0, tol_gap=1e-3, tol=1e-6,
                 tiempo_limite=1800.0, max_cortes_nodo=50, verbose=True,
                 x_inicial=None):
        self.inst = inst
        self.L = L
        self.tol_gap = tol_gap
        self.tol = tol
        self.tiempo_limite = tiempo_limite
        self.max_cortes_nodo = max_cortes_nodo
        self.verbose = verbose
        # Ruta factible con la que arrancar el incumbente. La relajación LP del
        # ruteo con MTZ es débil, así que entrar con una cota superior razonable
        # (por ejemplo la ruta del perfil promedio) reduce mucho la exploración.
        self.x_inicial = x_inicial
        self.registro = []

    def _es_binario(self, x_vec):
        return bool(np.all(np.minimum(x_vec, 1.0 - x_vec) <= 1e-6))

    def solve(self):
        t0 = time.time()
        maestro = MaestroRuteo(self.inst, theta_lb=self.L)
        evaluador = EvaluadorRecurso(self.inst)

        raiz = (np.zeros(self.inst.nA), np.ones(self.inst.nA))
        activos = [(-np.inf, 0, raiz)]
        contador, nodos = 1, 0
        UB, incumbente = np.inf, None
        LB = -np.inf
        motivo = "árbol agotado"

        if self.x_inicial is not None:
            x0 = np.round(np.asarray(self.x_inicial, dtype=float))
            Q0, corte0, _, _ = evaluador.evaluar(x0)
            UB = float(self.inst.c @ x0 + Q0)
            incumbente = x0
            maestro.agregar_corte_benders(corte0)
            maestro.agregar_corte_entero(x0, Q0, self.L)
            if self.verbose:
                print(f"  incumbente inicial: z = {UB:.4f}")

        while activos:
            if time.time() - t0 > self.tiempo_limite:
                motivo = "límite de tiempo"
                LB = activos[0][0] if activos else UB
                break

            cota, _, (lb_v, ub_v) = heapq.heappop(activos)
            LB = cota
            if np.isfinite(UB):
                gap = (UB - LB) / max(1.0, abs(UB))
                if gap <= self.tol_gap:
                    motivo = "tolerancia alcanzada"
                    break

            nodos += 1
            res = maestro.resolver(lb_v, ub_v)
            if res is None:
                continue
            x_vec, theta, obj_nodo = res
            if obj_nodo >= UB - self.tol:
                continue

            # Separación de cortes estándar antes de ramificar.
            Qk, corte, vigente = None, None, False
            for _ in range(self.max_cortes_nodo):
                Qk, corte, _, _ = evaluador.evaluar(x_vec)
                vigente = True  # Qk corresponde al x_vec actual
                if theta >= Qk - 1e-6 * max(1.0, abs(Qk)):
                    break
                maestro.agregar_corte_benders(corte)
                res = maestro.resolver(lb_v, ub_v)
                if res is None:
                    break
                x_vec, theta, obj_nodo = res
                vigente = False  # x_vec cambió: Qk quedó obsoleto
                if obj_nodo >= UB - self.tol:
                    break
            if res is None or obj_nodo >= UB - self.tol:
                continue

            if self._es_binario(x_vec):
                x_bin = np.round(x_vec)
                if not vigente:
                    Qk, corte, _, _ = evaluador.evaluar(x_bin)
                z = float(self.inst.c @ x_bin + Qk)
                if z < UB - self.tol:
                    UB, incumbente = z, x_bin.copy()
                    if self.verbose:
                        print(f"  nodo {nodos}: incumbente z = {z:.4f}")
                maestro.agregar_corte_entero(x_bin, Qk, self.L)
                maestro.agregar_corte_benders(corte)
                contador += 1
                heapq.heappush(activos, (obj_nodo, contador, (lb_v, ub_v)))
            else:
                k = int(np.argmin(np.abs(x_vec - 0.5)))
                lb0, ub0 = lb_v.copy(), ub_v.copy()
                ub0[k] = 0.0
                lb1, ub1 = lb_v.copy(), ub_v.copy()
                lb1[k] = 1.0
                contador += 1
                heapq.heappush(activos, (obj_nodo, contador, (lb0, ub0)))
                contador += 1
                heapq.heappush(activos, (obj_nodo, contador, (lb1, ub1)))

            gap_actual = ((UB - LB) / max(1.0, abs(UB))
                          if np.isfinite(UB) else np.inf)
            self.registro.append({
                "nodo": nodos,
                "tiempo": time.time() - t0,
                "LB": LB,
                "UB": UB,
                "gap": gap_actual,
                "cortes_benders": maestro.n_benders,
                "cortes_enteros": maestro.n_enteros,
                "cortes_total": maestro.n_benders + maestro.n_enteros,
            })
        else:
            LB = UB

        gap_final = ((UB - LB) / max(1.0, abs(UB))
                     if np.isfinite(UB) else np.inf)
        if self.verbose:
            print(f"Integer L-shaped: {motivo} | nodos = {nodos} | "
                  f"LB = {LB:.4f} | UB = {UB:.4f} | gap = {gap_final:.2e} | "
                  f"cortes = {maestro.n_benders}+{maestro.n_enteros}")

        return {
            "x": incumbente,
            "obj": UB,
            "LB": LB,
            "UB": UB,
            "gap": gap_final,
            "nodos": nodos,
            "motivo": motivo,
            "tiempo": time.time() - t0,
            "cortes_benders": maestro.n_benders,
            "cortes_enteros": maestro.n_enteros,
            "lps_resueltos": evaluador.n_lps,
            "registro": self.registro,
            "ruta": extraer_ruta(self.inst, incumbente) if incumbente is not None else None,
        }


# --------------------------------------------------------------------------
# Evaluación de una ruta fija sobre una muestra -- sin cambios de lógica.
# --------------------------------------------------------------------------
def evaluar_ruta(inst, x_vec, evaluador=None):
    """Calcula C_hat(x) = sum c_ij x_ij + (1/K) sum_s Q(x, xi_s) y diagnósticos.

    No reoptimiza la primera etapa: la ruta entra fija. Se usa tanto para el
    cálculo del VSS dentro de muestra como para la validación con febrero.
    """
    x_vec = np.asarray(x_vec, dtype=float)
    if evaluador is None:
        evaluador = EvaluadorRecurso(inst)
    Qk, _, Q_s, info = evaluador.evaluar(x_vec, detalle=True)

    costo_ruta = float(inst.c @ x_vec)
    o_s = np.array([d["o"] for d in info])
    e_s = np.array([d["e"] for d in info])
    r_s = np.vstack([d["r"] for d in info])

    return {
        "costo_total": costo_ruta + Qk,
        "costo_ruta": costo_ruta,
        "Q_prom": Qk,
        "costo_por_escenario": costo_ruta + Q_s,
        "Q_por_escenario": Q_s,
        "tau_por_escenario": np.array([d["tau"] for d in info]),
        "frec_overtime": float(np.mean(o_s > 1e-6)),
        "frec_tercerizacion": float(np.mean(r_s.sum(axis=1) > 1e-6)),
        "frec_emergencia": float(np.mean(e_s > 1e-6)),
        "tercerizacion_por_sitio": r_s.mean(axis=0),
        "costo_tercerizacion": float(np.mean(r_s @ np.array([C_OUT[i] for i in SITIOS]))),
        "costo_overtime": float(np.mean(C_OT * o_s)),
        "costo_emergencia": float(np.mean(C_EM * e_s)),
        "ruta": extraer_ruta(inst, x_vec),
    }


def comparar_rutas(inst, x_estrella, x_promedio):
    """Comparación pareada de dos rutas sobre los mismos escenarios."""
    evaluador = EvaluadorRecurso(inst)
    est = evaluar_ruta(inst, x_estrella, evaluador)
    prom = evaluar_ruta(inst, x_promedio, evaluador)

    D = prom["costo_por_escenario"] - est["costo_por_escenario"]
    n = len(D)
    media = float(D.mean())
    error = float(D.std(ddof=1) / np.sqrt(n)) if n > 1 else 0.0
    return {
        "x_estrella": est,
        "x_promedio": prom,
        "delta": prom["costo_total"] - est["costo_total"],
        "D_media": media,
        "D_error_estandar": error,
        "ic95": (media - 1.96 * error, media + 1.96 * error),
        "D": D,
    }

In [ ]:
%%writefile p1_experimentos.py
"""Orquestación de los experimentos de la Práctica 1 y visualizaciones.

Cubre la sección 5 del enunciado: prueba de consistencia con K = 50,
experimento principal con K = 200, aproximación del perfil promedio,
VSS dentro de muestra y validación temporal con febrero de 2015.
"""

import numpy as np

from p1_datos import ZONAS
# NOTA: único cambio respecto al p1_experimentos.py original -- se importa
# p1_modelo_scipy (scipy/HiGHS, sin licencia) en vez de p1_modelo (gurobipy).
# El resto de este archivo es idéntico; el API que usa es el mismo.
from p1_modelo_scipy import (
    C_OUT, SITIOS, EvaluadorRecurso, IntegerLShaped, Instancia, comparar_rutas,
    construir_instancia, evaluar_ruta, muestrear_escenarios, pools_a_numpy,
    resolver_extenso,
)

SEMILLA_ENTRENAMIENTO = 29
SEMILLA_PRUEBA = 2027


# --------------------------------------------------------------------------
# 5.1 Prueba de consistencia
# --------------------------------------------------------------------------
def prueba_consistencia(pools, K=50, semilla=SEMILLA_ENTRENAMIENTO,
                        tiempo_limite=1800, tol_relativa=1e-4):
    """Resuelve la MISMA muestra con la forma extensa y con Integer L-shaped."""
    inst = construir_instancia(pools, K=K, seed=semilla)

    ext = resolver_extenso(inst, tiempo_limite=tiempo_limite)
    calentamiento = resolver_extenso(inst.promedio(), tiempo_limite=tiempo_limite)
    ls = IntegerLShaped(inst, L=0.0, tol_gap=1e-4, tiempo_limite=tiempo_limite,
                        verbose=False, x_inicial=calentamiento["x"]).solve()

    rel = abs(ls["obj"] - ext["obj"]) / max(1.0, abs(ext["obj"]))
    print(f"Forma extensa    : {ext['obj']:.6f}  "
          f"({ext['n_vars']} vars, {ext['n_cons']} restr, {ext['tiempo']:.1f} s)")
    print(f"  ruta: {ext['ruta']}")
    print(f"Integer L-shaped : {ls['obj']:.6f}  "
          f"({ls['nodos']} nodos, {ls['cortes_benders']}+{ls['cortes_enteros']} cortes, "
          f"{ls['tiempo']:.1f} s)")
    print(f"  ruta: {ls['ruta']}")
    print(f"Diferencia relativa: {rel:.3e}  "
          f"({'COINCIDEN' if rel <= tol_relativa else 'NO COINCIDEN'} a {tol_relativa:g})")
    if ext["ruta"] != ls["ruta"]:
        print("  Nota: las rutas difieren; con costos iguales puede haber óptimos alternativos.")
    return {"instancia": inst, "extenso": ext, "lshaped": ls, "rel": rel}


# --------------------------------------------------------------------------
# 5.3 Experimento principal, perfil promedio y VSS
# --------------------------------------------------------------------------
def experimento_principal(pools, K=200, semilla=SEMILLA_ENTRENAMIENTO,
                          tiempo_limite=1800):
    inst = construir_instancia(pools, K=K, seed=semilla)

    # Se resuelve primero el perfil promedio: además de ser un entregable, su
    # ruta es un incumbente factible con el que arrancar el Integer L-shaped.
    print("Aproximación del perfil promedio (MILP determinista)")
    inst_prom = inst.promedio()
    prom = resolver_extenso(inst_prom, tiempo_limite=tiempo_limite)
    print(f"  x_prom: {prom['ruta']}")
    print(f"  objetivo determinista = {prom['obj']:.6f}  "
          f"({prom['n_vars']} vars, {prom['n_cons']} restr, {prom['tiempo']:.1f} s)")

    print(f"\nInteger L-shaped con K = {K} (límite {tiempo_limite / 60:.0f} min)")
    ls = IntegerLShaped(inst, L=0.0, tol_gap=1e-3, tiempo_limite=tiempo_limite,
                        verbose=True, x_inicial=prom["x"]).solve()
    print(f"  x*  : {ls['ruta']}")
    print(f"  LB = {ls['LB']:.6f}  UB = {ls['UB']:.6f}  gap = {ls['gap']:.3e}  "
          f"({ls['motivo']})")

    print("\nComparación justa sobre los MISMOS K escenarios (sin reoptimizar)")
    cmp = comparar_rutas(inst, ls["x"], prom["x"])
    print(f"  C_hat_K(x*)     = {cmp['x_estrella']['costo_total']:.6f}")
    print(f"  C_hat_K(x_prom) = {cmp['x_promedio']['costo_total']:.6f}")
    print(f"  VSS_K           = {cmp['delta']:.6f}")
    if ls["motivo"] != "límite de tiempo" and cmp["delta"] < -1e-6:
        print("  ATENCIÓN: VSS negativo con x* certificada; revisar.")

    return {"instancia": inst, "lshaped": ls, "promedio": prom,
            "comparacion": cmp, "x_estrella": ls["x"], "x_promedio": prom["x"]}


# --------------------------------------------------------------------------
# 5.3.4 Validación temporal fuera de muestra
# --------------------------------------------------------------------------
def validacion_febrero(pools_feb_alineados, c_entrenamiento, x_estrella,
                       x_promedio, K_test=1000, semilla=SEMILLA_PRUEBA):
    """Evalúa las dos rutas ya fijas sobre escenarios de febrero.

    Los costos c_ij y las distancias se mantienen en los valores de
    entrenamiento: febrero solo aporta los pools de tiempos de viaje.
    """
    arcos, _, pools_dict = pools_a_numpy(pools_feb_alineados)
    xi = muestrear_escenarios(arcos, pools_dict, K_test, semilla)
    inst = Instancia(arcos, c_entrenamiento, xi)

    cmp = comparar_rutas(inst, x_estrella, x_promedio)
    est, prom = cmp["x_estrella"], cmp["x_promedio"]

    print(f"Validación temporal con {K_test} escenarios de febrero de 2015")
    print(f"  C_hat_Feb(x*)     = {est['costo_total']:.6f}")
    print(f"  C_hat_Feb(x_prom) = {prom['costo_total']:.6f}")
    print(f"  Delta_Feb         = {cmp['delta']:.6f}")
    print(f"  D promedio        = {cmp['D_media']:.6f}  "
          f"(s.e. {cmp['D_error_estandar']:.6f})")
    print(f"  IC 95%            = ({cmp['ic95'][0]:.6f}, {cmp['ic95'][1]:.6f})")
    significativo = cmp["ic95"][0] > 0 or cmp["ic95"][1] < 0
    print(f"  El intervalo {'NO ' if not significativo else ''}excluye el cero: "
          f"{'hay' if significativo else 'no hay'} evidencia de diferencia.")

    print("\n  Frecuencias de activación del recurso")
    print(f"  {'ruta':<10}{'tiempo adic.':>14}{'terceriz.':>12}{'emergencia':>13}")
    for nombre, d in (("x*", est), ("x_prom", prom)):
        print(f"  {nombre:<10}{d['frec_overtime']:>14.3f}"
              f"{d['frec_tercerizacion']:>12.3f}{d['frec_emergencia']:>13.3f}")

    return {"instancia": inst, "comparacion": cmp, "significativo": significativo}


# --------------------------------------------------------------------------
# Tabla comparativa de enfoques
# --------------------------------------------------------------------------
def tabla_enfoques(consistencia, principal):
    ext, ls = consistencia["extenso"], consistencia["lshaped"]
    ls200, prom = principal["lshaped"], principal["promedio"]
    filas = [
        ("Forma extensa (K=50)", ext["obj"], ext["n_vars"], ext["n_cons"], ext["tiempo"]),
        ("Integer L-shaped (K=50)", ls["obj"], "maestro+K sub", "cortes dinámicos", ls["tiempo"]),
        ("Integer L-shaped (K=200)", ls200["obj"], "maestro+K sub", "cortes dinámicos", ls200["tiempo"]),
        ("Perfil promedio (MILP)", prom["obj"], prom["n_vars"], prom["n_cons"], prom["tiempo"]),
    ]
    ancho = max(len(f[0]) for f in filas) + 2
    print(f"{'Enfoque':<{ancho}}{'Objetivo':>14}{'Variables':>18}{'Restricciones':>18}{'Tiempo [s]':>12}")
    for nombre, obj, nv, nc, t in filas:
        print(f"{nombre:<{ancho}}{obj:>14.4f}{str(nv):>18}{str(nc):>18}{t:>12.2f}")
    print("\nNota: el objetivo del perfil promedio evalúa el recurso una sola vez en "
          "xi promedio\ny no es comparable directamente con los valores SAA.")
    return filas


# ==========================================================================
# Visualizaciones
# ==========================================================================
def _coords():
    lat = {z[0]: z[2] for z in ZONAS}
    lon = {z[0]: z[3] for z in ZONAS}
    nombre = {z[0]: z[1] for z in ZONAS}
    return lat, lon, nombre


def figura_mapa_rutas(ruta_estrella, ruta_promedio, ax=None):
    """Mapa comparativo de x* y x_prom con depósito, sitios y orden de visita."""
    import matplotlib.pyplot as plt

    lat, lon, nombre = _coords()
    iguales = ruta_estrella == ruta_promedio

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 9))

    for ruta, color, etiqueta, estilo, ancho in (
            (ruta_estrella, "#1f77b4", "x* (SAA)", "-", 2.2),
            (ruta_promedio, "#d62728", "x_prom (perfil promedio)", "--", 1.6)):
        if ruta is None:
            continue
        xs = [lon[n] for n in ruta]
        ys = [lat[n] for n in ruta]
        ax.plot(xs, ys, estilo, color=color, linewidth=ancho, label=etiqueta,
                alpha=0.85, zorder=2)

    for n in sorted(lat):
        es_deposito = n == 0
        ax.scatter(lon[n], lat[n], s=190 if es_deposito else 110,
                   marker="s" if es_deposito else "o",
                   color="black" if es_deposito else "white",
                   edgecolor="black", zorder=3)
        ax.annotate(str(n), (lon[n], lat[n]), ha="center", va="center",
                    fontsize=8, color="white" if es_deposito else "black", zorder=4)
        ax.annotate(nombre[n], (lon[n], lat[n]), textcoords="offset points",
                    xytext=(9, 7), fontsize=7, color="#444444", zorder=4)

    if iguales:
        ax.set_title("Rutas óptimas: x* y x_prom COINCIDEN", fontsize=11)
    else:
        ax.set_title("Rutas óptimas: x* frente a x_prom", fontsize=11)
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")
    ax.legend(loc="lower left", fontsize=8)
    ax.grid(alpha=0.25)
    return ax


def figura_pools(pools, arcos_destacados=None):
    """Matriz de tamaños de pools y distribuciones de al menos cuatro arcos."""
    import matplotlib.pyplot as plt

    n = 11
    tam = np.full((n, n), np.nan)
    dist = {}
    for fila in pools.iter_rows(named=True):
        i, j = int(fila["i"]), int(fila["j"])
        tam[i, j] = fila["registros"]
        dist[(i, j)] = np.asarray(fila["P_ij"], dtype=float)

    if arcos_destacados is None:
        ordenados = sorted(dist, key=lambda a: -len(dist[a]))
        arcos_destacados = ordenados[:2] + ordenados[-2:]

    fig = plt.figure(figsize=(13, 5.5))
    ax0 = fig.add_subplot(1, 2, 1)
    im = ax0.imshow(tam, cmap="viridis")
    ax0.set_title(f"Tamaño de los {int(np.isfinite(tam).sum())} pools |R_ij|", fontsize=10)
    ax0.set_xlabel("destino j")
    ax0.set_ylabel("origen i")
    ax0.set_xticks(range(n))
    ax0.set_yticks(range(n))
    fig.colorbar(im, ax=ax0, fraction=0.046, label="registros")

    ax1 = fig.add_subplot(1, 2, 2)
    for arco in arcos_destacados:
        ax1.hist(dist[arco], bins=30, histtype="step", linewidth=1.6,
                 density=True, label=f"({arco[0]},{arco[1]})  n={len(dist[arco])}")
    ax1.set_title("Distribución empírica del tiempo de viaje", fontsize=10)
    ax1.set_xlabel("minutos")
    ax1.set_ylabel("densidad")
    ax1.legend(fontsize=8)
    ax1.grid(alpha=0.25)
    fig.tight_layout()
    return fig


def figura_convergencia(registro):
    """Evolución de LB, UB, brecha y número acumulado de cortes."""
    import matplotlib.pyplot as plt

    if not registro:
        raise ValueError("El registro del algoritmo está vacío")
    t = np.array([r["tiempo"] for r in registro])
    lb = np.array([r["LB"] for r in registro], dtype=float)
    ub = np.array([r["UB"] for r in registro], dtype=float)
    gap = np.array([r["gap"] for r in registro], dtype=float)
    cortes = np.array([r["cortes_total"] for r in registro])

    finito = np.isfinite(lb)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

    axes[0].plot(t[finito], lb[finito], label="LB", color="#1f77b4")
    axes[0].plot(t[np.isfinite(ub)], ub[np.isfinite(ub)], label="UB", color="#d62728")
    axes[0].set_title("Cotas globales", fontsize=10)
    axes[0].set_xlabel("tiempo [s]")
    axes[0].set_ylabel("costo")
    axes[0].legend(fontsize=8)

    valido = np.isfinite(gap) & (gap > 0)
    axes[1].semilogy(t[valido], gap[valido], color="#2ca02c")
    axes[1].axhline(1e-3, color="gray", linestyle=":", label="tolerancia 1e-3")
    axes[1].set_title("Brecha relativa", fontsize=10)
    axes[1].set_xlabel("tiempo [s]")
    axes[1].legend(fontsize=8)

    axes[2].plot(t, cortes, color="#9467bd")
    axes[2].set_title("Cortes acumulados", fontsize=10)
    axes[2].set_xlabel("tiempo [s]")
    axes[2].set_ylabel("cortes")

    for ax in axes:
        ax.grid(alpha=0.25)
    fig.tight_layout()
    return fig


def figura_comparacion(est, prom, tiempos=None):
    """Costos, frecuencias de recurso y tiempos de cómputo de los enfoques."""
    import matplotlib.pyplot as plt

    n_paneles = 3 if tiempos else 2
    fig, axes = plt.subplots(1, n_paneles, figsize=(5.2 * n_paneles, 4.2))

    etiquetas = ["x*", "x_prom"]
    ancho = 0.35
    pos = np.arange(2)
    componentes = [
        ("ruta", [est["costo_ruta"], prom["costo_ruta"]], "#4c72b0"),
        ("tercerización", [est["costo_tercerizacion"], prom["costo_tercerizacion"]], "#dd8452"),
        ("tiempo adicional", [est["costo_overtime"], prom["costo_overtime"]], "#55a868"),
        ("emergencia", [est["costo_emergencia"], prom["costo_emergencia"]], "#c44e52"),
    ]
    base = np.zeros(2)
    for nombre, vals, color in componentes:
        axes[0].bar(pos, vals, ancho * 1.6, bottom=base, label=nombre, color=color)
        base += np.array(vals)
    axes[0].set_xticks(pos)
    axes[0].set_xticklabels(etiquetas)
    axes[0].set_title("Composición del costo esperado", fontsize=10)
    axes[0].set_ylabel("USD")
    axes[0].legend(fontsize=8)

    frec = [("tiempo adic.", "frec_overtime"), ("terceriz.", "frec_tercerizacion"),
            ("emergencia", "frec_emergencia")]
    pos2 = np.arange(len(frec))
    axes[1].bar(pos2 - ancho / 2, [est[k] for _, k in frec], ancho, label="x*")
    axes[1].bar(pos2 + ancho / 2, [prom[k] for _, k in frec], ancho, label="x_prom")
    axes[1].set_xticks(pos2)
    axes[1].set_xticklabels([n for n, _ in frec], fontsize=8)
    axes[1].set_ylim(0, 1.05)
    axes[1].set_title("Frecuencia de activación del recurso", fontsize=10)
    axes[1].legend(fontsize=8)

    if tiempos:
        nombres = list(tiempos)
        axes[2].barh(range(len(nombres)), [tiempos[n] for n in nombres], color="#8172b3")
        axes[2].set_yticks(range(len(nombres)))
        axes[2].set_yticklabels(nombres, fontsize=8)
        axes[2].set_xlabel("segundos")
        axes[2].set_title("Tiempo de cómputo", fontsize=10)

    for ax in axes:
        ax.grid(alpha=0.25, axis="both")
    fig.tight_layout()
    return fig


def figura_fuera_muestra(est, prom, D):
    """Composición del costo fuera de muestra y tercerización por sitio."""
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

    pos = np.arange(len(SITIOS))
    ancho = 0.38
    axes[0].bar(pos - ancho / 2, est["tercerizacion_por_sitio"], ancho, label="x*")
    axes[0].bar(pos + ancho / 2, prom["tercerizacion_por_sitio"], ancho, label="x_prom")
    axes[0].set_xticks(pos)
    axes[0].set_xticklabels(SITIOS)
    axes[0].set_xlabel("sitio i")
    axes[0].set_ylabel("minutos promedio")
    axes[0].set_title("Tercerización promedio por sitio (febrero)", fontsize=10)
    axes[0].legend(fontsize=8)

    ax2 = axes[0].twinx()
    ax2.plot(pos, [C_OUT[i] for i in SITIOS], "k.--", linewidth=1, markersize=7,
             label="c_out")
    ax2.set_ylabel("c_out [USD/min]", fontsize=8)
    ax2.legend(fontsize=7, loc="upper right")

    axes[1].hist(est["costo_por_escenario"], bins=40, alpha=0.6, label="x*")
    axes[1].hist(prom["costo_por_escenario"], bins=40, alpha=0.6, label="x_prom")
    axes[1].set_xlabel("costo por escenario [USD]")
    axes[1].set_title("Distribución del costo fuera de muestra", fontsize=10)
    axes[1].legend(fontsize=8)

    media = float(np.mean(D))
    error = float(np.std(D, ddof=1) / np.sqrt(len(D)))
    axes[2].hist(D, bins=40, color="#937860")
    axes[2].axvline(0, color="black", linestyle=":", linewidth=1)
    axes[2].axvline(media, color="#c44e52", linewidth=1.8,
                    label=f"media {media:.3f}")
    axes[2].axvspan(media - 1.96 * error, media + 1.96 * error, color="#c44e52",
                    alpha=0.2, label="IC 95%")
    axes[2].set_xlabel("D = C(x_prom) - C(x*)")
    axes[2].set_title("Diferencias pareadas", fontsize=10)
    axes[2].legend(fontsize=8)

    for ax in axes:
        ax.grid(alpha=0.25)
    fig.tight_layout()
    return fig

In [ ]:
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt

import p1_datos as datos
import p1_modelo_scipy as modelo
import p1_experimentos as exp

SEMILLA_ENTRENAMIENTO = 29
SEMILLA_PRUEBA = 2027
K_CONSISTENCIA = 50
K_PRINCIPAL = 200
K_TEST = 1000
LIMITE_SEGUNDOS = 30 * 60

print("Semillas:", SEMILLA_ENTRENAMIENTO, SEMILLA_PRUEBA)

# Formulación matemática

## Conjuntos y datos

- $V=\{0,1,\dots,10\}$ nodos, $N=V\setminus\{0\}$ sitios, $A=\{(i,j)\in V\times V: i\neq j\}$, $|A|=110$.
- $c_{ij}=\kappa\, d^{\text{road}}_{ij}$ con $\kappa=1$ USD/km y $d^{\text{road}}_{ij}=1.60934\times\text{mediana}\{\texttt{trip\_distance}_r\}$.
- $d_i$ demanda de trabajo [min], $c^{\text{out}}_i$ costo unitario de tercerización [USD/min].
- $H=390$, $\bar o=30$, $c^{OT}=1.50$, $c^{EM}=6.00$, $\alpha_i=1$.
- $\xi^{(s)}_{ij}$: tiempo de viaje del arco $(i,j)$ en el escenario $s$, muestreado con reemplazo del pool $P_{ij}$.

## Primera etapa

Variables: $x_{ij}\in\{0,1\}$ indica si la ruta usa el arco $(i,j)$; $v_i\in[1,|N|]$ continuas son las variables de orden de Miller–Tucker–Zemlin.

$$\min_{x,v}\ \sum_{(i,j)\in A} c_{ij}x_{ij} + \mathcal{Q}_K(x)$$

sujeto a

$$\sum_{j\neq i} x_{ij}=1 \quad \forall i\in V \qquad \text{(sale una vez de cada nodo)}$$
$$\sum_{i\neq j} x_{ij}=1 \quad \forall j\in V \qquad \text{(entra una vez a cada nodo)}$$
$$v_i-v_j+|N|\,x_{ij}\le |N|-1 \quad \forall i,j\in N,\ i\neq j \qquad \text{(MTZ: elimina subtours)}$$
$$1\le v_i\le |N| \quad \forall i\in N, \qquad x_{ij}\in\{0,1\}.$$

Las restricciones de grado por sí solas admiten varios ciclos disjuntos. MTZ asigna a cada sitio una posición $v_i$ en el recorrido y obliga a que $v_j\ge v_i+1$ cuando se usa el arco $(i,j)$ entre sitios, lo que hace imposible cerrar un ciclo que no pase por el depósito.

## Segunda etapa (continua)

Para la ruta $x$ fija y el escenario $\xi_s$:

$$Q(x,\xi_s)=\min_{u,r,o,e}\ \sum_{i\in N} c^{\text{out}}_i r^{(s)}_i + c^{OT}o^{(s)} + c^{EM}e^{(s)}$$

sujeto a

$$u^{(s)}_i + r^{(s)}_i = d_i \quad \forall i\in N \qquad [\pi^{(s)}_i] \qquad \text{(toda la demanda se cubre)}$$
$$\sum_{(i,j)\in A}\xi^{(s)}_{ij}x_{ij} + \sum_{i\in N}\alpha_i u^{(s)}_i \le H + o^{(s)} + e^{(s)} \qquad [\lambda_s\le 0] \qquad \text{(presupuesto de tiempo)}$$
$$0\le u^{(s)}_i\le d_i,\quad r^{(s)}_i\ge 0,\quad 0\le o^{(s)}\le\bar o,\quad e^{(s)}\ge 0.$$

La aproximación por promedio muestral es $\mathcal{Q}_K(x)=\frac{1}{K}\sum_{s=1}^K Q(x,\xi_s)$.

Como $e^{(s)}$ no tiene cota superior, el recurso es **completo**: el subproblema es factible para toda ruta y todo escenario. Por lo tanto no se requieren cortes de factibilidad, y como todos los costos de recurso son no negativos, $L=0$ es una cota inferior global válida.

## Forma extensa MILP

Se replican las variables de segunda etapa $K$ veces y se resuelve un único modelo:

$$\min\ \sum_{(i,j)\in A} c_{ij}x_{ij} + \frac{1}{K}\sum_{s=1}^{K}\Big(\sum_{i\in N} c^{\text{out}}_i r^{(s)}_i + c^{OT}o^{(s)} + c^{EM}e^{(s)}\Big)$$

sujeto a las restricciones de ruteo y, para cada $s$, a las de segunda etapa.

## Propiedad estructural

El escenario entra en el subproblema únicamente a través del escalar

$$\tau_s(x)=\sum_{(i,j)\in A}\xi^{(s)}_{ij}x_{ij},$$

de modo que $Q(x,\xi_s)=\varphi(\tau_s(x))$ para una **única** función convexa lineal por tramos $\varphi$, la misma para todos los escenarios. **En esta versión (scipy) esta propiedad ya no es solo una verificación: es el motor de cálculo.** `p1_modelo_scipy.py` evalúa $\mathcal{Q}_K(x)$ y los cortes de Benders directamente con la forma cerrada $\varphi$ (función `recurso_analitico`), sin resolver ningún LP por escenario.

## Cortes del Integer L-shaped

**Corte estándar de Benders**, agregado con probabilidad $1/K$. De la desigualdad de subgradiente $Q_s(\tau)\ge Q_s(\tau^\nu)-\lambda_s(\tau-\tau^\nu)$:

$$\theta \ \ge\ \frac{1}{K}\sum_{s}\big(Q(x^\nu,\xi_s)+\lambda_s\tau_s(x^\nu)\big)\ -\ \frac{1}{K}\sum_{s}\lambda_s\sum_{(i,j)\in A}\xi^{(s)}_{ij}x_{ij}.$$

**Corte de optimalidad entero** (Laporte y Louveaux) con $L=0$ y $S=\{(i,j):x^\nu_{ij}=1\}$:

$$\theta \ \ge\ \big(\mathcal{Q}_K(x^\nu)-L\big)\Big(\sum_{(i,j)\in S}x_{ij}-\sum_{(i,j)\notin S}x_{ij}-|S|+1\Big)+L.$$

Es ajustado en $x^\nu$ y se reduce a $\theta\ge L$ en cualquier otro punto binario. Ambos cortes son **globales**: se acumulan en un único maestro compartido por todos los nodos del árbol.

## Esquema del algoritmo

1. Seleccionar el nodo activo de **mejor cota**.
2. Resolver el maestro **LP** del nodo: ruteo relajado, $\theta\ge L$, cotas locales de ramificación y todos los cortes globales.
3. Podar si el nodo es infactible o si su cota alcanza el incumbente.
4. **Separación de cortes**: mientras $\theta$ subestime $\mathcal{Q}_K(x^\nu)$, agregar el corte de Benders violado y reoptimizar el nodo.
5. Si $x^\nu$ es **binaria**: evaluar el recurso exacto, actualizar el incumbente y agregar el corte de optimalidad entero.
6. Si es **fraccional**: ramificar sobre la variable de arco más cercana a $0.5$.
7. Parar cuando $\text{gap}=(UB-LB)/\max\{1,|UB|\}\le 10^{-3}$.

## 1. Datos de entrenamiento: enero de 2015

Se aplican los cuatro filtros del enunciado. El límite superior de la ventana de
fechas es **exclusivo** (`2015-02-01`), de modo que el último día del mes queda
incluido.

In [ ]:
archivo_enero = datos.descargar(datos.URL_ENERO)
registros_enero, pools_enero = datos.construir_pools(
    archivo_enero, datetime(2015, 1, 1),
    datetime(2015, 2, 1))

print(f"Registros tras los filtros: {len(registros_enero):,}")
pools_enero.head()

In [ ]:
# Control de calidad: deben aparecer 110 pools y el más pequeño 87 registros.
datos.control_calidad(pools_enero, n_arcos_esperado=110, min_registros_esperado=87)

In [ ]:
fig = exp.figura_pools(pools_enero)
plt.show()

## 2. Prueba de consistencia con K = 50

La **misma** muestra se resuelve de dos maneras y los valores objetivo deben
coincidir dentro de una tolerancia relativa de $10^{-4}$. No se exige que las
rutas coincidan si existen óptimos alternativos.

In [ ]:
consistencia = exp.prueba_consistencia(
    pools_enero, K=K_CONSISTENCIA, semilla=SEMILLA_ENTRENAMIENTO,
    tiempo_limite=LIMITE_SEGUNDOS)

### Verificación independiente del recurso

Como $Q(x,\xi_s)=\varphi(\tau_s)$, la forma cerrada $\varphi$ (`recurso_analitico`)
es lo que usa `EvaluadorRecurso` internamente en esta versión -- ya no resuelve
un LP por escenario. Para que la verificación sea realmente independiente (no
circular), esta celda arma el LP del subproblema a mano con
`scipy.optimize.linprog`, para una muestra de escenarios, y lo compara contra
la forma cerrada, valores y duales.

In [ ]:
from scipy.optimize import linprog


def _lp_subproblema(tau):
    """LP explícito del subproblema de segunda etapa (referencia independiente
    de la forma cerrada que usa EvaluadorRecurso)."""
    n = modelo.N_SITIOS
    c = np.zeros(2 * n + 2)
    for t, i in enumerate(modelo.SITIOS):
        c[n + t] = modelo.C_OUT[i]
    c[2 * n] = modelo.C_OT
    c[2 * n + 1] = modelo.C_EM

    A_eq = np.zeros((n, 2 * n + 2))
    b_eq = np.zeros(n)
    for t, i in enumerate(modelo.SITIOS):
        A_eq[t, t] = 1.0
        A_eq[t, n + t] = 1.0
        b_eq[t] = modelo.DEMANDA[i]

    A_ub = np.zeros((1, 2 * n + 2))
    for t, i in enumerate(modelo.SITIOS):
        A_ub[0, t] = modelo.ALPHA[i]
    A_ub[0, 2 * n] = -1.0
    A_ub[0, 2 * n + 1] = -1.0
    b_ub = np.array([modelo.H - tau])

    bounds = ([(0, None)] * n + [(0, None)] * n
              + [(0, modelo.O_BAR)] + [(0, None)])
    return linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                    bounds=bounds, method="highs")


inst50 = consistencia["instancia"]
ev = modelo.EvaluadorRecurso(inst50)
_, _, Q_cerrada, info = ev.evaluar(consistencia["lshaped"]["x"], detalle=True)
tau = np.array([d["tau"] for d in info])

rng_verif = np.random.default_rng(0)
muestra = rng_verif.choice(inst50.K, size=min(15, inst50.K), replace=False)
err_Q, err_dual = [], []
for s in muestra:
    res = _lp_subproblema(tau[s])
    Q_c, marginal_c = modelo.recurso_analitico(tau[s])
    err_Q.append(abs(res.fun - Q_c[0]))
    err_dual.append(abs(res.ineqlin.marginals[0] - (-marginal_c[0])))

print(f"max |Q_LP - Q_cerrada|          = {max(err_Q):.3e}  "
      f"(sobre {len(muestra)} escenarios resueltos como LP independiente)")
print(f"max |lambda_LP - (-marginal)|   = {max(err_dual):.3e}")

## 3. Experimento principal (K = 200) y perfil promedio

Se resuelve primero la aproximación determinista del perfil promedio: además de
ser un entregable, su ruta sirve como incumbente inicial del Integer L-shaped.
Luego se comparan ambas rutas **sobre los mismos 200 escenarios**, sin
reoptimizar ninguna de las dos.

In [ ]:
principal = exp.experimento_principal(
    pools_enero, K=K_PRINCIPAL, semilla=SEMILLA_ENTRENAMIENTO,
    tiempo_limite=LIMITE_SEGUNDOS)

x_estrella = principal["x_estrella"]
x_promedio = principal["x_promedio"]

In [ ]:
fig = exp.figura_convergencia(principal["lshaped"]["registro"])
plt.show()

In [ ]:
ax = exp.figura_mapa_rutas(principal["lshaped"]["ruta"], principal["promedio"]["ruta"])
plt.show()

## 4. Validación temporal fuera de muestra: febrero de 2015

Se repite el procedimiento de construcción de pools con las fechas de febrero,
**sin mezclarlos** con los de enero. Las rutas $x^\star$ y $x_{\text{prom}}$, las
distancias y los costos $c_{ij}$ quedan fijos: febrero solo aporta nuevos
escenarios de tiempos de viaje.

In [ ]:
archivo_febrero = datos.descargar(datos.URL_FEBRERO)
registros_febrero, pools_febrero = datos.construir_pools(
    archivo_febrero, datetime(2015, 2, 1),
    datetime(2015, 3, 1))

# Control de calidad: 110 pares dirigidos y el pool más pequeño con 90 registros.
datos.control_calidad(pools_febrero, n_arcos_esperado=110, min_registros_esperado=90)

In [ ]:
# Se alinea el orden de arcos y se conservan los costos de entrenamiento.
pools_febrero_alineados = datos.alinear_pools(pools_enero, pools_febrero)
_, c_entrenamiento, _ = modelo.pools_a_numpy(pools_enero)

validacion = exp.validacion_febrero(
    pools_febrero_alineados, c_entrenamiento, x_estrella, x_promedio,
    K_test=K_TEST, semilla=SEMILLA_PRUEBA)

In [ ]:
est = validacion["comparacion"]["x_estrella"]
prom = validacion["comparacion"]["x_promedio"]
fig = exp.figura_fuera_muestra(est, prom, validacion["comparacion"]["D"])
plt.show()

## 5. Comparación de enfoques

In [ ]:
filas = exp.tabla_enfoques(consistencia, principal)

In [ ]:
fig = exp.figura_comparacion(est, prom, tiempos={
    "extensa K=50": consistencia["extenso"]["tiempo"],
    "L-shaped K=50": consistencia["lshaped"]["tiempo"],
    "L-shaped K=200": principal["lshaped"]["tiempo"],
    "perfil promedio": principal["promedio"]["tiempo"],
})
plt.show()

### Exportar las figuras para el informe

Guarda las cinco visualizaciones con los nombres que espera `informe/informe.tex`
y las comprime para descargarlas.

In [ ]:
import os, zipfile

os.makedirs("figuras", exist_ok=True)

exportables = {
    "fig_pools.png": exp.figura_pools(pools_enero),
    "fig_convergencia.png": exp.figura_convergencia(principal["lshaped"]["registro"]),
    "fig_mapa.png": exp.figura_mapa_rutas(principal["lshaped"]["ruta"],
                                          principal["promedio"]["ruta"]).figure,
    "fig_fuera_muestra.png": exp.figura_fuera_muestra(
        est, prom, validacion["comparacion"]["D"]),
    "fig_comparacion.png": exp.figura_comparacion(est, prom, tiempos={
        "extensa K=50": consistencia["extenso"]["tiempo"],
        "L-shaped K=50": consistencia["lshaped"]["tiempo"],
        "L-shaped K=200": principal["lshaped"]["tiempo"],
        "perfil promedio": principal["promedio"]["tiempo"],
    }),
}
for nombre, fig in exportables.items():
    fig.savefig(os.path.join("figuras", nombre), dpi=200, bbox_inches="tight")
    plt.close(fig)

with zipfile.ZipFile("figuras_informe.zip", "w") as z:
    for nombre in exportables:
        z.write(os.path.join("figuras", nombre))

print("Figuras guardadas en figuras/ y comprimidas en figuras_informe.zip")
try:
    from google.colab import files
    files.download("figuras_informe.zip")
except Exception:
    pass

## 6. Preguntas de interpretación

> Las respuestas se desarrollan en el informe técnico en PDF. Las celdas de esta
> sección producen la evidencia numérica que las sustenta.

1. **Compromiso entre distancia determinista y exposición a tiempos altos.**
2. **Escenarios que activan la tercerización y sitios que la concentran.**
3. **Qué aporta la evaluación fuera de muestra frente al valor óptimo SAA.**
4. **Qué se pierde al usar zonas TLC y al muestrear los arcos de forma independiente.**
5. **Por qué $\bar{\mathcal{Q}}_K(x)=Q(x,\bar\xi_K)$ y $\mathcal{Q}_K(x)$ difieren** (convexidad y desigualdad de Jensen).
6. **Comparación de $x^\star$ y $x_{\text{prom}}$ con $VSS_K$, $\Delta_{\text{Feb}}$ y el intervalo pareado.**

In [ ]:
# Evidencia para la pregunta 5: desigualdad de Jensen.
inst200 = principal["instancia"]
ev200 = modelo.EvaluadorRecurso(inst200)
Q_saa, _, _, _ = ev200.evaluar(x_estrella)
Q_prom_perfil, _, _, _ = modelo.EvaluadorRecurso(inst200.promedio()).evaluar(x_estrella)

print(f"Q_K(x*)            = {Q_saa:.6f}   (promedio de los K recursos)")
print(f"Q(x*, xi_promedio) = {Q_prom_perfil:.6f}   (recurso en el perfil promedio)")
print(f"Diferencia (Jensen) = {Q_saa - Q_prom_perfil:.6f}  (debe ser >= 0 por convexidad)")

In [ ]:
# Evidencia para la pregunta 2: dónde se concentra la tercerización.
orden = np.argsort(-est["tercerizacion_por_sitio"])
print(f"{'sitio':<8}{'min. tercerizados':>20}{'c_out':>10}{'demanda':>10}")
for k in orden:
    i = modelo.SITIOS[k]
    print(f"{i:<8}{est['tercerizacion_por_sitio'][k]:>20.3f}"
          f"{modelo.C_OUT[i]:>10.2f}{modelo.DEMANDA[i]:>10.0f}")

In [ ]:
# Evidencia para la pregunta 1: costo de ruta frente a exposición temporal.
for nombre, d in (("x*", est), ("x_prom", prom)):
    tau = d["tau_por_escenario"]
    print(f"{nombre:<8} costo ruta = {d['costo_ruta']:8.3f} km/USD | "
          f"tau medio = {tau.mean():7.2f} min | "
          f"p90 = {np.percentile(tau, 90):7.2f} | Q = {d['Q_prom']:8.3f}")